# Distributed PPO — **learner: train on the LATEST GCS rollout**

This notebook is the GPU **learner** side. It does NOT bootstrap or seed a run — it
**assumes rollouts already exist on GCS** and trains on the **newest completed one**
(`rollouts/v<K>/<actor>/_DONE`, picked by max update-time), producing `policy_v{K+1}`.
Cell §6 optionally flips the run back to `await_rollout` so the actor VMs roll the next
iter (they self-clean `/tmp` shards + `/dev/shm` between slices). Re-run to grab the next rollout.

## 0. Config — simple first try

In [ ]:
import time

# --- identity: control doc (Firestore) + big-file root (GCS) ---
PROJECT = 'analog-receiver-489214-e9'        # gcloud + Firestore both need an explicit project
BUCKET  = 'gs://orbit-wars-shipping'
RUN_ID  = 'ppo_gcp_' + time.strftime('%Y%m%d-%H%M%S')
PREFIX  = f'{BUCKET}/ppo/{RUN_ID}'           # gs:// RUN ROOT for big files (checkpoints/heads/rollouts)
STATE_URL = 'fs://ppo_runs/' + RUN_ID        # Firestore CONTROL doc (the authoritative STATE)

# --- loop shape (tiny: a first smoke of the full ping-pong) ---
ITERS          = 2             # total rollout->train iterations (the state machine's iters_total)
EPOCHS         = 8             # PPO epochs per learner_step
EPISODES       = 6             # self-play games per rollout, per GCP actor VM
NUM_PLAYERS    = 'mix'         # 'mix' = alternate 2P/4P (distinct seeds); or '2' / '4'
HISTORY_WINDOW = 10            # T=10 rollout window
SEED_BASE      = 100000        # per-actor distinct seed slice: the daemon offsets by
                               # iter/actor-index off SEED_BASE -> fresh, non-overlapping games
MAX_PLANETS    = 32
MAX_FLEETS     = 256           # rollout shard fleet cap (actual fleets << this; keeps shards light)
SIGMA          = 0.35          # frac logit-normal sigma (fixed)
DEVICE         = 'cuda'        # LEARNER device (this Colab). The rollout actors run device='cpu'.

# --- multi-VM actor roster + COW pool backend ---
# EXPECTED_ACTORS: each id rolls a DISTINCT, non-overlapping slice of EPISODES games
# per iter; the phase only flips await_rollout->await_train once EVERY id has written
# its rollouts/vK/<id>/_DONE (the gated barrier). Single-actor by default; add ids to
# fan the rollout across machines.
EXPECTED_ACTORS = ['c4-0']                 # multi e.g. ['c4-0','n2-0']
POOL_PROCS      = 8            # COW fork-pool size per actor: ONE model load, forked across
                              # N cores (one model in RAM, not N) — the preferred actor path

# --- PPO hyperparams (learner) ---
MINIBATCH   = 128  # max_planets=32 -> pair head 4x smaller -> 128 fits the 40GB GPU
MAX_SHARDS  = 32   # cap shards in RAM (stride-sampled); full 64 OOMs Colab host RAM
CLIP        = 0.20
TARGET_KL   = 0.10   # per-component; LOOSE for early training (tighten later)
LR_HEADS    = 3e-5            # lr for the action heads (+critic ValueDecoder)
LR_TRUNK    = 1e-5            # SMALLER lr for the unfrozen trunk/L3/L4 — protects the
                             # supervised backbone from PPO wrecking it at Phase 2
VALUE_COEF  = 0.5
ENT_COEF    = 0.01
BC_COEF     = 0.0              # behavior-clone anchor OFF for the first try

# --- unfreeze schedule: Phase 2 unfreezes heads + trunk/FiLM/L4 + L3 dual_role + critic.
# The learner trains this set; v0's head-delta MUST be saved with the SAME freeze set so
# its key-set matches (see §3). LR_TRUNK (above) gates the trunk/L3/L4 lr.
FREEZE_PHASE = 2

# --- frozen base (actor backbone); critic = post-L2 ValueDecoder trained in PPO ---
BASELINE_RUN = 'L3L4_T10_film3_head3_d256_b256_20ep_lr0.0001_20260529-084115'
ENTITY_CKPT  = f'{BUCKET}/entity/runs/{BASELINE_RUN}/entity_encoder_best.pt'

LEARNER_ID = 'colab'           # this side's id stamped into STATE.learner

print('='*78)
print('DISTRIBUTED PPO — SIMPLE FIRST TRY')
print('='*78)
print(f'  RUN_ID        {RUN_ID}')
print(f'  STATE_URL     {STATE_URL}      (Firestore control doc)')
print(f'  PREFIX        {PREFIX}   (gs:// big-file root)')
print(f'  PROJECT       {PROJECT}')
print(f'  loop          iters={ITERS}  episodes/iter/actor={EPISODES}  '
      f'players={NUM_PLAYERS}  T={HISTORY_WINDOW}')
print(f'  actors        expected={EXPECTED_ACTORS}  pool_procs={POOL_PROCS} (COW pool/actor)')
print(f'  devices       learner={DEVICE}   |   actor VM(s)=cpu')
print(f'  seeds/caps    seed_base={SEED_BASE}  max_planets={MAX_PLANETS}  '
      f'max_fleets={MAX_FLEETS}  sigma={SIGMA}')
print(f'  PPO           epochs={EPOCHS}  mb={MINIBATCH}  clip={CLIP}  kl={TARGET_KL}  '
      f'vc={VALUE_COEF}  ec={ENT_COEF}  bc={BC_COEF}')
print(f'  unfreeze      freeze_phase={FREEZE_PHASE} (heads+trunk/L4+L3+critic)  '
      f'lr_heads={LR_HEADS}  lr_trunk={LR_TRUNK}')
print(f'  baseline      {BASELINE_RUN}')
print(f'  entity ckpt   {ENTITY_CKPT}')
print(f'  learner id    {LEARNER_ID}')
print('='*78)

## 1. Authenticate + set project + Firestore client

In [ ]:
from google.colab import auth
auth.authenticate_user()
import subprocess, sys, os
subprocess.run(['gcloud','config','set','project',PROJECT], check=True)   # REQUIRED for any gcloud
# The LEARNER daemon writes the Firestore STATE doc from THIS Colab, so the client lib
# must be present here and the project must be on the env (the firestore.Client + the
# daemons both read PPO_FIRESTORE_PROJECT).
subprocess.run([sys.executable,'-m','pip','install','-q','google-cloud-firestore'], check=True)
os.environ['PPO_FIRESTORE_PROJECT'] = PROJECT

def gcs(*a): return subprocess.run(['gcloud','storage',*a], check=True, capture_output=True, text=True)
def gcs_exists(url):
    try: gcs('objects','describe',url,'--format=value(size)'); return True
    except subprocess.CalledProcessError: return False

def run_streaming(cmd):
    # subprocess.run's inherited stdout does NOT stream into a Colab cell -- it
    # surfaces only when the child exits, so a long rollout/train looks "stuck".
    # Read the child's stdout via a pipe and re-print each line at the Python level
    # so the rollout heartbeats + learner signals show LIVE.
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

print('project set:', PROJECT)
print('PPO_FIRESTORE_PROJECT:', os.environ['PPO_FIRESTORE_PROJECT'])
print('google-cloud-firestore installed (learner writes the STATE doc from here)')

## 2. Stage code + weights + actor ckpt

In [ ]:
import shutil
subprocess.run([sys.executable,'-m','pip','install','-q','kaggle_environments','psutil'], check=True)
from pathlib import Path
WORK = Path('/content/orbit-wars'); WORK.mkdir(parents=True, exist_ok=True); os.chdir(WORK)
for rel in ('agents','scripts','ckpts'): shutil.rmtree(WORK/rel, ignore_errors=True)
# entity/ has the FLAT weights.tgz (planet/fleet/comet) + the ppo-package code.tgz
gcs('cp', f'{BUCKET}/entity/code.tgz', '.');    subprocess.run(['tar','xzf','code.tgz'], check=True)
gcs('cp', f'{BUCKET}/entity/weights.tgz', '.'); subprocess.run(['tar','xzf','weights.tgz'], check=True)
PLANET_DIR=WORK/'ckpts/planet'; FLEET_DIR=WORK/'ckpts/fleet'; COMET_DIR=WORK/'ckpts/comet'; ENT_DIR=WORK/'ckpts/entity'
for d in (PLANET_DIR,FLEET_DIR,COMET_DIR,ENT_DIR): d.mkdir(parents=True, exist_ok=True)
for src,dst in (('planet_encoder_best.pt',PLANET_DIR),('fleet_encoder_best.pt',FLEET_DIR),('comet_past_best.pt',COMET_DIR)):
    if (WORK/src).exists(): shutil.copy(WORK/src, dst/src)
ENTITY_LOCAL=str(ENT_DIR/'entity_encoder_best.pt'); gcs('cp', ENTITY_CKPT, ENTITY_LOCAL)
for m in [m for m in sys.modules if m.startswith('agents')]: del sys.modules[m]
import agents; from agents.transformer_v2.ppo import shards
import torch; print('agents at', agents.__file__, '| cuda:', torch.cuda.is_available())
print('staged ckpts/{planet,fleet,comet,entity}:',
      [p.name for p in (PLANET_DIR,FLEET_DIR,COMET_DIR,ENT_DIR)])

## 3. Base-version (consume existing run — no policy_v0 bootstrap)

In [ ]:
# ── 3. Base-version stamp (NO bootstrap) ──────────────────────────────────────
# This notebook CONSUMES rollouts already on GCS, so we do NOT build/push a fresh
# policy_v0 here — the run's policy_vK checkpoints already exist on GCS (written by
# whoever bootstrapped/trained it). We only need BASE_VERSION (hash of the frozen
# entity backbone) to stamp the head-delta learner_step writes.
from agents.transformer_v2.ppo import shards
BASE_VERSION = shards.sha256_file(ENTITY_LOCAL)[:12]
print('base_version', BASE_VERSION, '| entity', ENTITY_LOCAL)

## 4. Discover the latest rollout on GCS

In [ ]:
# ── 4. Pick the rollout to train on ──────────────────────────
# Default: the LATEST completed rollout on GCS (newest rollouts/v<K>/<actor>/_DONE
# by update-time). To RE-TRAIN a specific (e.g. the previous) rollout instead —
# handy while iterating on KL/lr — set PIN_ITER to its K (e.g. 1 -> rollouts/v1/).
# learner_step ALWAYS overwrites policy_v{K+1}, so re-running re-trains cleanly.
PIN_ITER = None   # None = latest ; e.g. 1 = force-train rollouts/v1/ -> policy_v2
from google.cloud import storage as _gcs

def find_latest_rollout(bucket_name=BUCKET.replace("gs://", ""), ppo_prefix="ppo/", pin_iter=None):
    cl = _gcs.Client(project=PROJECT)
    dones = []  # (updated, run_id, K, vK, actor)
    for bl in cl.list_blobs(bucket_name, prefix=ppo_prefix):
        n = bl.name
        if not (n.endswith("/_DONE") and "/rollouts/v" in n):
            continue
        p = n.split("/")
        try:
            ri = p.index("rollouts"); run_id = "/".join(p[1:ri]); vK = p[ri + 1]; K = int(vK[1:]); actor = p[ri + 2]
        except (ValueError, IndexError):
            continue
        dones.append((bl.updated, run_id, K, vK, actor))
    if pin_iter is not None:
        dones = [x for x in dones if x[2] == pin_iter]
    if not dones:
        return None
    dones.sort()
    _, run_id, K, vK, _ = dones[-1]
    run_prefix = f"{BUCKET}/{ppo_prefix}{run_id}"
    actors = sorted({x[4] for x in dones if x[1] == run_id and x[2] == K})
    n_shards = sum(1 for b in cl.list_blobs(bucket_name, prefix=f"{ppo_prefix}{run_id}/rollouts/{vK}/")
                   if "/shard_" in b.name and b.name.endswith(".pt"))
    return dict(run_id=run_id, K=K, vK=vK, run_prefix=run_prefix,
                state_url=f"fs://ppo_runs/{run_id}",
                shards_url=f"{run_prefix}/rollouts/{vK}/",
                policy_ckpt=f"{run_prefix}/checkpoints/policy_v{K}.pt",
                out_ckpt=f"{run_prefix}/checkpoints/policy_v{K + 1}.pt",
                out_heads=f"{run_prefix}/heads/policy_v{K + 1}.heads.pt",
                n_shards=n_shards, actors=actors)

LR = find_latest_rollout(pin_iter=PIN_ITER)
assert LR, "no completed rollout found (check PIN_ITER / GCS under ppo/)"
print("TRAINING ON ROLLOUT")
print("  run_id    :", LR["run_id"])
print("  iter K    :", LR["K"], " ->  policy_v%d -> policy_v%d  (OVERWRITES out_ckpt)" % (LR["K"], LR["K"] + 1))
print("  actors    :", LR["actors"], "| shards:", LR["n_shards"])
print("  shards    :", LR["shards_url"])
print("  out_ckpt  :", LR["out_ckpt"])


## 5. Train on the latest rollout (learner_step → policy_v{K+1})

In [ ]:
# ── 5. Train on the LATEST rollout — learner_step (one-shot) -> policy_v{K+1} ──
# Re-run this whole notebook whenever a newer rollout lands on GCS; find_latest_rollout
# above always picks the freshest one. (Idempotency: if out_ckpt already exists you are
# re-training the same data — bump past it by rolling a new iter first.)
run_streaming(['python', '-u', '-m', 'agents.transformer_v2.ppo.learner_step',
    '--policy-ckpt',    LR['policy_ckpt'],
    '--shards',         LR['shards_url'],
    '--out-ckpt',       LR['out_ckpt'],
    '--out-heads',      LR['out_heads'],
    '--policy-version', str(LR['K']),
    '--base-version',   BASE_VERSION,
    '--ckpt',           ENTITY_LOCAL,
    '--fleet-run-dir',  str(FLEET_DIR), '--planet-run-dir', str(PLANET_DIR),
    '--comet-run-dir',  str(COMET_DIR),
    '--device',         'cuda',
    '--epochs',         str(EPOCHS),
    '--freeze-phase',   str(FREEZE_PHASE),                      # heads+trunk/L4+L3+critic
    '--lr-heads',       str(LR_HEADS), '--lr-trunk', str(LR_TRUNK),
    '--minibatch-size', str(MINIBATCH), '--clip', str(CLIP), '--target-kl', str(TARGET_KL),
    '--value-coef',     str(VALUE_COEF), '--ent-coef', str(ENT_COEF),
    '--sigma',          str(SIGMA),
    '--max-shards', str(MAX_SHARDS),
    '--train-log',      f"{LR['run_prefix']}/train_log.jsonl"])
print('\ntrained: policy_v%d -> policy_v%d  ->  %s' % (LR['K'], LR['K'] + 1, LR['out_ckpt']))

## 6. (optional) Advance the loop — let the actors roll the next iter

In [ ]:
# ── 6. OPTIONAL — advance the loop ────────────────────────────────────────────
# Flip THIS run's STATE to await_rollout iter K+1 with the freshly-trained
# policy_v{K+1}. The actor daemons (parked at await_train, polling every 10s) pick
# it up and roll the next iteration with the new policy — and clean themselves
# (/tmp shards + /dev/shm) after each slice. SKIP this cell if you only wanted the
# trained checkpoint and don't want to trigger more rollouts.
from agents.transformer_v2.ppo import ppo_state as S
_nxt = LR['K'] + 1
S.transition(LR['state_url'], expect_phase=None, new_phase=S.PHASE_AWAIT_ROLLOUT,
             who='colab-learner', iter=_nxt,
             model=S.model_block(LR['run_prefix'], _nxt, BASE_VERSION))
print('advanced %s -> await_rollout iter %d (model=policy_v%d).' % (LR['state_url'], _nxt, _nxt))
print('listening actor daemons will roll v%d on their next poll.' % _nxt)

## 6. Monitor (optional live dashboard)

A read-only streaming dashboard over the same STATE doc: phase / iter + whichever side is active
and its live progress + log tail. Runnable here, or from Colab or any rollout VM simultaneously (STATE is the one
shared truth). It exits when the run reaches `done`. Run this in a **second** Colab tab if you want
a clean dashboard while §5 does the training.

In [ ]:
run_streaming(['python','-u','-m','agents.transformer_v2.ppo.ppo_monitor','--state', LR['state_url']])

## 7. Inspect — PPO training signals + game status (per iter)

In [ ]:
import json
print('=== PPO TRAINING SIGNALS (per iter, from train_log.jsonl) ===')
raw = subprocess.run(['gcloud','storage','cat',f'{PREFIX}/train_log.jsonl'],capture_output=True,text=True).stdout
for ln in raw.splitlines():
    r=json.loads(ln)
    print(f"iter {r['iter']:>2}: winrate={r.get('winrate')}  kl={r.get('avg_kl')}  "
          f"policy_loss={r.get('policy_loss')}  value_loss={r.get('value_loss')}  "
          f"entropy={r.get('entropy')}  clip_frac={r.get('clip_frac')}  "
          f"(eps={r.get('n_episodes')} steps={r.get('n_steps')} wall={r.get('wall_s')}s)")

print('\n=== GAME STATUS (per iter: win rate, 2P/4P split, game length, RAM/timing) ===')
# GCP actors write shards under rollouts/vK/<actor_id>/ — discover the id dir(s) rather than assume it.
for K in range(ITERS):
    base = f'{PREFIX}/rollouts/v{K}/'
    m = None
    for entry in shards.gcs_list(base):
        cand = entry.rstrip('/') + '/metrics.json'
        mm = shards.read_progress(cand)
        if mm: m = mm; break
    if not m: continue
    pe = m.get('per_episode',[])
    g2=[e for e in pe if e['num_players']==2]; g4=[e for e in pe if e['num_players']==4]
    w2=sum(e['won'] for e in g2); w4=sum(e['won'] for e in g4)
    steps=[e['steps'] for e in pe] or [0]
    split=[]
    if g2: split.append(f"2P {w2}/{len(g2)}={w2/len(g2)*100:.0f}%")
    if g4: split.append(f"4P {w4}/{len(g4)}={w4/len(g4)*100:.0f}%")
    print(f"v{K}: {len(pe)} games, {m.get('n_wins')}/{len(pe)} wins  [{', '.join(split)}]  "
          f"game-len mean {sum(steps)/len(steps):.0f} (min {min(steps)} max {max(steps)})  | "
          f"peak_rss {m.get('peak_rss_mb')}MB  mean_step {m.get('mean_step_s')}s  wall {m.get('total_wall_s')}s")